# Лабораторная работа 5. Дерево решений

Цель — самостоятельно реализовать дерево решений (CART) для задачи классификации и сравнить его с `DecisionTreeClassifier` из sklearn.

> **Подсказка: что такое дерево решений.**
> Дерево решений — это модель в виде схемы из вопросов «да/нет». Объект («клиент» или «строка таблицы») входит в **корень** (верхний узел), отвечает на вопрос вида «признак ≤ порога?» и идёт влево (да) или вправо (нет). Так он спускается по **узлам** (`Node`), пока не попадёт в **лист** (`Leaf`) — конечную вершину с готовым ответом (классом).
> - **Узел** хранит: номер признака (`index`), порог (`t`) и два поддерева (`true_branch` — условие выполнено, `false_branch` — нет).
> - **Лист** хранит объекты, которые в него попали, и прогноз — самый частый класс среди них.
> - **Обучение дерева** — это подбор вопросов: в каждом узле перебираем все признаки и пороги и выбираем тот вопрос, который лучше всего разделяет классы. Затем так же строим поддеревья для каждой части (рекурсия).
> - В этой лабе мы пишем всё по шагам: измерить «перемешанность» классов (Джини, задание 1), оценить пользу вопроса (прирост информации, задание 2), решить, когда остановиться (задание 3), оценить качество (задание 4) и сравнить с готовым деревом из sklearn (задание 5).

## Заготовка из задания
Класс узла дерева и класс листа (терминального узла).

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_SEED = 42


# Класс узла
class Node:
    def __init__(self, index, t, true_branch, false_branch):
        self.index = index  # индекс признака, по которому ведется сравнение с порогом в этом узле
        self.t = t  # значение порога
        self.true_branch = true_branch  # поддерево, удовлетворяющее условию в узле
        self.false_branch = false_branch  # поддерево, не удовлетворяющее условию в узле


# Класс терминального узла (листа)
class Leaf:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        # подсчет количества объектов разных классов
        classes = {}  # словарь "класс: количество объектов"
        for label in self.labels:
            if label not in classes:
                classes[label] = 0
            classes[label] += 1
        # класс с максимальным количеством объектов в этом листе;
        # при равенстве выбираем класс с меньшей меткой (как argmax в sklearn)
        best = max(classes.values())
        prediction = min(label for label, count in classes.items() if count == best)
        return prediction

### Задание 1
Реализовать функцию расчёта критерия Джини.

Критерий Джини (неопределённость, impurity) множества объектов с метками классов:

$$G = 1 - \\sum_{k} p_k^2,$$

где $p_k$ — доля объектов класса $k$ в узле. Для «чистого» узла (один класс) $G = 0$; для двух равных по размеру классов $G = 0.5$ (максимум для бинарной задачи).

> **Подсказка к заданию 1: критерий Джини.**
> - **Что измеряем.** Насколько «перемешаны» классы в узле. Если в узле только один класс, всё понятно и неопределённости нет: Джини = 0. Если классов поровну (50/50), перемешанность максимальная: для двух классов Джини = 0.5.
> - **Формула** G = 1 − Σ pₖ², где pₖ — доля класса k в узле. Пример: в узле 3 объекта класса 0 и 1 объект класса 1, значит доли 0.75 и 0.25, G = 1 − (0.75² + 0.25²) = 0.375.
> - **Как работает код.** `np.unique(..., return_counts=True)` считает, сколько объектов каждого класса, деление на общее число даёт доли `probs`, а `1 - np.sum(probs ** 2)` — сама формула.
> - **Проверки в ноутбуке** — это случаи, где ответ можно посчитать в уме. Если функция их даёт верно, ей можно доверять.

In [2]:
# Расчёт критерия Джини
def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / len(labels)          # доли классов p_k
    return 1.0 - np.sum(probs ** 2)

Проверка на простых примерах, для которых результат можно посчитать вручную.

In [3]:
tests = {
    'один класс [1, 1, 1, 1]': ([1, 1, 1, 1], 0.0),
    'поровну [0, 0, 1, 1]': ([0, 0, 1, 1], 0.5),
    '3 к 1 [0, 0, 0, 1]': ([0, 0, 0, 1], 1 - (0.75 ** 2 + 0.25 ** 2)),
    'три класса поровну [0, 1, 2]': ([0, 1, 2], 1 - 3 * (1 / 3) ** 2),
    'пустой массив []': ([], 0.0),
}
for name, (labels, expected) in tests.items():
    value = gini(labels)
    assert np.isclose(value, expected)
    print(f'{name:32s} G = {value:.4f}  (ожидается {expected:.4f})')

один класс [1, 1, 1, 1]          G = 0.0000  (ожидается 0.0000)
поровну [0, 0, 1, 1]             G = 0.5000  (ожидается 0.5000)
3 к 1 [0, 0, 0, 1]               G = 0.3750  (ожидается 0.3750)
три класса поровну [0, 1, 2]     G = 0.6667  (ожидается 0.6667)
пустой массив []                 G = 0.0000  (ожидается 0.0000)


Строки и числовые метки классов обрабатываются одинаково, так как функция опирается только на частоты классов.

In [4]:
print(gini(np.array(['a', 'b', 'a', 'a'])), gini(np.array([5, 7, 5, 5])))

0.375 0.375


**Итог по заданию 1.** Функция `gini` возвращает $1 - \\sum p_k^2$ по частотам классов, корректно работает для пустого узла, двух и более классов и для нечисловых меток. Она нужна для оценки неопределённости узла и расчёта прироста информации (задание 2).

### Задание 2
Реализовать расчёт критерия прироста информации.

Прирост информации при разбиении узла на левую и правую части — это уменьшение неопределённости (Джини) после разбиения:

$$Gain = G_{root} - \frac{n_{left}}{n}\,G_{left} - \frac{n_{right}}{n}\,G_{right},$$

где $n = n_{left} + n_{right}$. Чем больше прирост, тем лучше разбиение; для разбиения, не изменяющего долю классов, $Gain = 0$.

> **Подсказка к заданию 2: прирост информации.**
> - **Что измеряем.** Насколько вопрос в узле «навёл порядок». Берём Джини до разбиения (`root_gini`) и вычитаем среднюю (взвешенную по числу объектов) перемешанность двух частей после разбиения. Чем больше разница, тем полезнее вопрос.
> - **Формула** Gain = G_корня − (n_лев/n)·G_лев − (n_прав/n)·G_прав. Веса n_лев/n и n_прав/n нужны, чтобы большая часть влияла сильнее маленькой.
> - **Крайние случаи.** Идеальный вопрос полностью разделяет классы: обе части чистые, Gain = G_корня. Бесполезный вопрос оставляет в обеих частях те же доли классов, что были, и Gain = 0.
> - **`split`** — вспомогательная функция из заготовки. Она делит данные по правилу «значение признака ≤ порога» на левую и правую части.
> - **Демонстрация на 6 объектах** показывает: перебирая пороги, находим тот, где Gain максимален (там классы разделились полностью).

In [5]:
# Расчёт прироста информации
def gain(left_labels, right_labels, root_gini):
    n_left, n_right = len(left_labels), len(right_labels)
    n = n_left + n_right
    if n == 0:
        return 0.0
    weighted = (n_left / n) * gini(left_labels) + (n_right / n) * gini(right_labels)
    return root_gini - weighted

Функция разбиения набора данных в узле из заготовки задания (понадобится при построении дерева): объекты с `data[:, column_index] <= t` попадают в левую ветку.

In [6]:
# Разбиение датасета в узле (из задания)
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)
    true_data = data[left]
    false_data = data[right]
    true_labels = labels[left]
    false_labels = labels[right]
    return true_data, false_data, true_labels, false_labels

Проверка на примерах с известным ответом.

In [7]:
labels = np.array([0, 0, 0, 0, 1, 1, 1, 1])
root = gini(labels)                                   # 0.5

# 1) идеальное разбиение: оба класса полностью разделены -> прирост равен G_root
print('идеальное:   ', gain(labels[:4], labels[4:], root))
assert np.isclose(gain(labels[:4], labels[4:], root), root)

# 2) бесполезное разбиение: в обеих частях те же доли классов -> прирост 0
print('бесполезное: ', gain(labels[[0, 1, 4, 5]], labels[[2, 3, 6, 7]], root))
assert np.isclose(gain(labels[[0, 1, 4, 5]], labels[[2, 3, 6, 7]], root), 0)

# 3) частичное разбиение, посчитанное вручную:
#    left = [0,0,0,1] (G=0.375), right = [0,1,1,1] (G=0.375) -> Gain = 0.5 - 0.375 = 0.125
print('частичное:   ', gain(labels[[0, 1, 2, 4]], labels[[3, 5, 6, 7]], root))
assert np.isclose(gain(labels[[0, 1, 2, 4]], labels[[3, 5, 6, 7]], root), 0.125)

# 4) пустая сторона разбиения (порог за границей данных) -> прирост 0
print('пустая часть:', gain(labels, labels[:0], root))
assert np.isclose(gain(labels, labels[:0], root), 0)

идеальное:    0.5
бесполезное:  0.0
частичное:    0.125
пустая часть: 0.0


Проверим связку `split` + `gain` на небольшом наборе: один признак, порог по значению признака.

In [8]:
X_demo = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y_demo = np.array([0, 0, 0, 1, 1, 1])
g_root = gini(y_demo)

for t in [1.0, 2.0, 3.0, 4.0, 5.0]:
    _, _, y_left, y_right = split(X_demo, y_demo, 0, t)
    print(f'порог {t}: слева {y_left.tolist()}, справа {y_right.tolist()}, Gain = {gain(y_left, y_right, g_root):.4f}')

порог 1.0: слева [0], справа [0, 0, 1, 1, 1], Gain = 0.1000
порог 2.0: слева [0, 0], справа [0, 1, 1, 1], Gain = 0.2500
порог 3.0: слева [0, 0, 0], справа [1, 1, 1], Gain = 0.5000
порог 4.0: слева [0, 0, 0, 1], справа [1, 1], Gain = 0.2500
порог 5.0: слева [0, 0, 0, 1, 1], справа [1], Gain = 0.1000


**Итог по заданию 2.** Функция `gain` вычисляет уменьшение Джини после разбиения как разность между неопределённостью родительского узла и взвешенной (по размерам частей) неопределённостью дочерних. Проверки подтверждают граничные случаи: идеальное разбиение даёт максимальный прирост, равный $G_{root}$, бесполезное и пустое — ноль. На демонстрационном наборе лучший порог (между 3 и 4) даёт полное разделение классов и максимальный прирост 0.5.

### Задание 3
Реализовать критерии останова (не менее двух).

В заготовке дерево растёт, пока есть прирост информации, то есть до полностью «чистых» листьев: это приводит к переобучению. Реализуем четыре критерия останова, которые задаются параметрами `build_tree`:

1. `max_depth` — максимальная глубина дерева;
2. `min_samples_split` — минимальное число объектов в узле, при котором узел ещё делится;
3. `min_samples_leaf` — минимальное число объектов в каждой из двух частей разбиения (листе);
4. `min_gain` — минимальный прирост информации, при котором разбиение допустимо.

Кроме того, узел не делится, если он уже «чистый» (все объекты одного класса, $G = 0$).

Два отличия от заготовки (оба нужны, чтобы результат совпал с sklearn):
- кандидаты в пороги — не сами значения признака, а середины между соседними различными значениями (так делает sklearn); иначе объекты, попадающие между значениями обучающей выборки, классифицировались бы по-разному;
- разбиение с нулевым приростом допустимо (как в sklearn), если узел не чистый, поэтому условие остановки `gain == 0` из заготовки заменено на `gain < min_gain`.

> **Подсказка к заданию 3: критерии останова и построение дерева.**
> - **Зачем останавливаться.** Если не ограничивать, дерево будет делить данные, пока в каждом листе не останется один объект. Оно «выучит» каждую точку, включая шум, и плохо сработает на новых данных. Это **переобучение**.
> - **`find_best_split`** перебирает все признаки и все пороги, считает прирост информации для каждого и возвращает лучший. Кандидаты в пороги — середины между соседними значениями признака.
> - **`build_tree`** строит дерево рекурсией: находит лучшее разбиение, делит данные на две части и вызывает сама себя для каждой. Рекурсия заканчивается, когда срабатывает один из критериев останова — тогда создаётся лист.
> - **Четыре критерия.** `max_depth` — максимальная глубина; `min_samples_split` — узел с меньшим числом объектов не делим; `min_samples_leaf` — в каждой части после разбиения должно быть не меньше стольких объектов; `min_gain` — не делим, если польза разбиения меньше порога. Плюс не делим «чистый» узел (все объекты одного класса).
> - **Как читать таблицу.** Сравните число листьев: без ограничений дерево получается огромным (68 листьев), а любое ограничение делает его компактнее.

In [9]:
# Нахождение наилучшего разбиения
def find_best_split(data, labels, min_samples_leaf=1):
    root_gini = gini(labels)
    best_gain = -1.0
    best_t = None
    best_index = None
    n_features = data.shape[1]
    for index in range(n_features):
        values = np.unique(data[:, index])
        # кандидаты в пороги: середины между соседними различными значениями
        for t in (values[:-1] + values[1:]) / 2:
            true_data, false_data, true_labels, false_labels = split(data, labels, index, t)
            # критерий останова: в каждой части должно быть не меньше min_samples_leaf объектов
            if len(true_labels) < min_samples_leaf or len(false_labels) < min_samples_leaf:
                continue
            current_gain = gain(true_labels, false_labels, root_gini)
            if current_gain > best_gain:
                best_gain, best_t, best_index = current_gain, t, index
    return best_gain, best_t, best_index


# Построение дерева с помощью рекурсивной функции
def build_tree(data, labels, depth=0, max_depth=None, min_samples_split=2, min_samples_leaf=1, min_gain=0.0):
    # критерий останова 1: достигнута максимальная глубина
    if max_depth is not None and depth >= max_depth:
        return Leaf(data, labels)
    # критерий останова 2: в узле слишком мало объектов для разбиения или узел уже чистый
    if len(labels) < min_samples_split or gini(labels) == 0:
        return Leaf(data, labels)

    best_gain, t, index = find_best_split(data, labels, min_samples_leaf)
    # критерий останова 3: не нашлось допустимого разбиения (min_samples_leaf)
    # критерий останова 4: прирост информации меньше min_gain
    if index is None or best_gain < min_gain:
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)
    # Рекурсивно строим два поддерева
    params = dict(max_depth=max_depth, min_samples_split=min_samples_split,
                  min_samples_leaf=min_samples_leaf, min_gain=min_gain)
    true_branch = build_tree(true_data, true_labels, depth + 1, **params)
    false_branch = build_tree(false_data, false_labels, depth + 1, **params)
    # Возвращаем узел со всеми поддеревьями, то есть целое дерево
    return Node(index, t, true_branch, false_branch)


def classify_object(obj, node):
    # Останавливаем рекурсию, если достигли листа
    if isinstance(node, Leaf):
        return node.prediction
    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    else:
        return classify_object(obj, node.false_branch)


def predict(data, tree):
    return np.array([classify_object(obj, tree) for obj in data])


# Напечатаем ход нашего дерева
def print_tree(node, spacing=""):
    # Если лист, то выводим его прогноз
    if isinstance(node, Leaf):
        print(spacing + "Прогноз:", node.prediction)
        return
    # Выведем значение индекса и порога на этом узле
    print(spacing + 'Индекс', str(node.index), '<=', str(round(node.t, 4)))
    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")
    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")


def tree_depth(node):
    if isinstance(node, Leaf):
        return 0
    return 1 + max(tree_depth(node.true_branch), tree_depth(node.false_branch))


def count_leaves(node):
    if isinstance(node, Leaf):
        return 1
    return count_leaves(node.true_branch) + count_leaves(node.false_branch)

Для проверки сгенерируем датасет бинарной классификации (`make_classification`): 500 объектов, 6 признаков, из них 4 информативных, 5% шума в метках. Разделим на обучающую и тестовую выборки (75/25, со стратификацией).

In [10]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=500, n_features=6, n_informative=4, n_redundant=1,
                           n_classes=2, class_sep=1.0, flip_y=0.05, random_state=DATA_SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, ' Test:', X_test.shape, ' Доли классов в train:', np.bincount(y_train) / len(y_train))

Train: (375, 6)  Test: (125, 6)  Доли классов в train: [0.49866667 0.50133333]


Влияние критериев останова на размер дерева: без ограничений дерево растёт до чистых листьев, а каждый критерий его останавливает раньше.

In [11]:
configs = {
    'без ограничений': dict(),
    'max_depth=3': dict(max_depth=3),
    'min_samples_split=50': dict(min_samples_split=50),
    'min_samples_leaf=20': dict(min_samples_leaf=20),
    'min_gain=0.02': dict(min_gain=0.02),
}
rows = []
for name, kw in configs.items():
    tree = build_tree(X_train, y_train, **kw)
    rows.append({'критерий': name, 'глубина': tree_depth(tree), 'листьев': count_leaves(tree)})
pd.DataFrame(rows)

,критерий,глубина,листьев
0,без ограничений,11,68
1,max_depth=3,3,8
2,min_samples_split=50,7,19
3,min_samples_leaf=20,5,13
4,min_gain=0.02,11,55


**Итог по заданию 3.** Реализованы четыре критерия останова (`max_depth`, `min_samples_split`, `min_samples_leaf`, `min_gain`) и остановка на чистом узле. Таблица показывает, что полное дерево сильно разрастается, а любое из ограничений делает его компактнее.

### Задание 4
Реализовать функцию подсчёта метрики модели.

В качестве метрики используем точность (accuracy) — долю правильных ответов, как предлагается в заготовке. Дополнительно реализуем F1-меру по положительному классу: она полезна, когда классы несбалансированы.

> **Подсказка к заданию 4: метрика качества.**
> - **Accuracy** (точность) — доля объектов, для которых предсказание совпало с правдой: `np.sum(actual == predicted) / len(actual)`. Просто, но при несбалансированных классах может обманывать.
> - **F1-мера** — компромисс между precision (доля верных среди предсказанных «положительных») и recall (доля найденных среди всех настоящих «положительных»). Для её подсчёта нужны три числа: TP (верно найденные положительные), FP (ложные срабатывания), FN (пропущенные положительные).
> - **Сверка с sklearn.** Мы проверяем свои функции на маленьком примере: `accuracy_score` и `f1_score` из sklearn должны дать то же число. Так мы убеждаемся, что формулы реализованы верно.

In [12]:
# Подсчёт точности как доли правильных ответов
def accuracy_metric(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    return np.sum(actual == predicted) / len(actual)


# F1-мера по положительному классу (дополнительная метрика)
def f1_metric(actual, predicted, positive=1):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    tp = np.sum((predicted == positive) & (actual == positive))
    fp = np.sum((predicted == positive) & (actual != positive))
    fn = np.sum((predicted != positive) & (actual == positive))
    if tp == 0:
        return 0.0
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)

In [13]:
from sklearn.metrics import accuracy_score, f1_score

actual    = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 1])
predicted = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0])
print('accuracy:', accuracy_metric(actual, predicted), ' sklearn:', accuracy_score(actual, predicted))
print('F1:      ', round(f1_metric(actual, predicted), 4), ' sklearn:', round(f1_score(actual, predicted), 4))
assert np.isclose(accuracy_metric(actual, predicted), accuracy_score(actual, predicted))
assert np.isclose(f1_metric(actual, predicted), f1_score(actual, predicted))

accuracy: 0.7  sklearn: 0.7
F1:       0.7273  sklearn: 0.7273


**Итог по заданию 4.** Функции `accuracy_metric` и `f1_metric` дают те же значения, что и `accuracy_score`/`f1_score` из sklearn.

### Задание 5
Проверить работу самописного дерева для решения задачи классификации (набор данных сгенерирован `make_classification`), сравнив результаты с деревом решений из sklearn. Результаты должны совпасть полностью.

Обучим оба дерева с одинаковыми параметрами (критерий Джини) и сравним: (1) предсказания на обучающей и тестовой выборках, (2) метрики, (3) структуру дерева (признак и порог в каждом узле). Пороги sklearn хранит в `float32`, поэтому пороги сравниваем с допуском $10^{-5}$.

> **Подсказка к заданию 5: сравнение с sklearn.**
> - **Идея.** Алгоритм построения дерева определён однозначно: перебрать разбиения, выбрать лучшее по Джини, повторять. Поэтому корректно написанное самописное дерево с теми же параметрами должно дать то же самое, что `DecisionTreeClassifier`. Если не совпало, значит, где-то ошибка.
> - **Что именно сравниваем.** Предсказания на обучающей и тестовой выборках, метрики и **структуру** дерева (в каждом узле тот же признак и тот же порог).
> - **Почему пороги сравниваем с допуском.** sklearn хранит их в `float32` (меньше знаков), поэтому числа отличаются в 8-м знаке, хотя по смыслу одинаковы.
> - **`make_classification`** создаёт искусственный набор для классификации, поэтому нам не нужны внешние файлы.
> - **Проверка на 20 наборах** нужна, чтобы убедиться, что совпадение не случайность на одном датасете.
> - **Оговорка про «ничьи».** Если два разных разбиения дают абсолютно одинаковый прирост, sklearn выбирает одно из них случайно, а наше дерево берёт первое найденное. Поэтому для сравнения выбраны параметры, где такие ситуации редки (`min_samples_leaf` ≥ 10).

In [14]:
from sklearn.tree import DecisionTreeClassifier, export_text

def my_structure(node, out=None):
    out = [] if out is None else out
    if isinstance(node, Leaf):
        out.append(('leaf', int(node.prediction)))
    else:
        out.append(('node', node.index, float(node.t)))
        my_structure(node.true_branch, out)
        my_structure(node.false_branch, out)
    return out

def sk_structure(model):
    tr, out = model.tree_, []
    def rec(i):
        if tr.children_left[i] == -1:
            out.append(('leaf', int(np.argmax(tr.value[i][0]))))
        else:
            out.append(('node', int(tr.feature[i]), float(tr.threshold[i])))
            rec(tr.children_left[i]); rec(tr.children_right[i])
    rec(0)
    return out

def same_structure(a, b):
    if len(a) != len(b):
        return False
    for p, q in zip(a, b):
        if p[:2] != q[:2]:
            return False
        if p[0] == 'node' and not np.isclose(p[2], q[2], rtol=1e-5, atol=1e-5):
            return False
    return True

settings = [
    dict(max_depth=3, min_samples_leaf=10),
    dict(max_depth=5, min_samples_leaf=20),
    dict(min_samples_leaf=20),
    dict(max_depth=6, min_samples_split=40, min_samples_leaf=15),
]
rows = []
for kw in settings:
    my_tree = build_tree(X_train, y_train, **kw)
    sk_tree = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE, **kw).fit(X_train, y_train)
    pred_my_te, pred_sk_te = predict(X_test, my_tree), sk_tree.predict(X_test)
    pred_my_tr, pred_sk_tr = predict(X_train, my_tree), sk_tree.predict(X_train)
    rows.append({
        'параметры': str(kw),
        'accuracy test (свое)': accuracy_metric(y_test, pred_my_te),
        'accuracy test (sklearn)': accuracy_metric(y_test, pred_sk_te),
        'F1 test (свое)': round(f1_metric(y_test, pred_my_te), 4),
        'F1 test (sklearn)': round(f1_metric(y_test, pred_sk_te), 4),
        'предсказания совпали': bool((pred_my_te == pred_sk_te).all() and (pred_my_tr == pred_sk_tr).all()),
        'структура совпала': same_structure(my_structure(my_tree), sk_structure(sk_tree)),
        'листьев': count_leaves(my_tree),
    })
comparison = pd.DataFrame(rows)
assert comparison['предсказания совпали'].all() and comparison['структура совпала'].all()
comparison

,параметры,accuracy test (свое),accuracy test (sklearn),F1 test (свое),F1 test (sklearn),предсказания совпали,структура совпала,листьев
0,"{'max_depth': 3, 'min_samples_leaf': 10}",0.664,0.664,0.6818,0.6818,True,True,7
1,"{'max_depth': 5, 'min_samples_leaf': 20}",0.720,0.720,0.7368,0.7368,True,True,13
2,{'min_samples_leaf': 20},0.720,0.720,0.7368,0.7368,True,True,13
3,"{'max_depth': 6, 'min_samples_split': 40, 'min...",0.728,0.728,0.7424,0.7424,True,True,14


Для наглядности выведем структуру обоих деревьев для первого набора параметров.

In [15]:
kw = settings[0]
my_tree = build_tree(X_train, y_train, **kw)
sk_tree = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE, **kw).fit(X_train, y_train)
print('=== Самописное дерево ===')
print_tree(my_tree)
print('\n=== sklearn ===')
print(export_text(sk_tree, feature_names=[f'Индекс {i}' for i in range(X.shape[1])]))

=== Самописное дерево ===
Индекс 5 <= -0.6178
--> True:
  Индекс 5 <= -2.4072
  --> True:
    Индекс 3 <= -1.1013
    --> True:
      Прогноз: 0
    --> False:
      Прогноз: 0
  --> False:
    Индекс 4 <= 1.8152
    --> True:
      Прогноз: 0
    --> False:
      Прогноз: 1
--> False:
  Индекс 3 <= 1.7417
  --> True:
    Индекс 1 <= -0.2605
    --> True:
      Прогноз: 1
    --> False:
      Прогноз: 1
  --> False:
    Прогноз: 0

=== sklearn ===
|--- Индекс 5 <= -0.62
|   |--- Индекс 5 <= -2.41
|   |   |--- Индекс 3 <= -1.10
|   |   |   |--- class: 0
|   |   |--- Индекс 3 >  -1.10
|   |   |   |--- class: 0
|   |--- Индекс 5 >  -2.41
|   |   |--- Индекс 4 <= 1.82
|   |   |   |--- class: 0
|   |   |--- Индекс 4 >  1.82
|   |   |   |--- class: 1
|--- Индекс 5 >  -0.62
|   |--- Индекс 3 <= 1.74
|   |   |--- Индекс 1 <= -0.26
|   |   |   |--- class: 1
|   |   |--- Индекс 1 >  -0.26
|   |   |   |--- class: 1
|   |--- Индекс 3 >  1.74
|   |   |--- class: 0



Проверим устойчивость: повторим сравнение на 20 разных наборах данных (разные `random_state` генератора) для нескольких параметров.

In [16]:
n_checked, n_ok = 0, 0
for seed in range(20):
    Xs, ys = make_classification(n_samples=500, n_features=6, n_informative=4, n_redundant=1,
                                 class_sep=1.0, flip_y=0.05, random_state=seed)
    Xa, Xb, ya, yb = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=RANDOM_STATE)
    for kw in [dict(min_samples_leaf=20), dict(max_depth=5, min_samples_leaf=20)]:
        mt = build_tree(Xa, ya, **kw)
        sk = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE, **kw).fit(Xa, ya)
        n_checked += 1
        n_ok += bool((predict(Xa, mt) == sk.predict(Xa)).all() and (predict(Xb, mt) == sk.predict(Xb)).all())
print(f'Совпали предсказания в {n_ok} из {n_checked} проверок')

Совпали предсказания в 40 из 40 проверок


**Итог по заданию 5.** Самописное дерево совпало с `DecisionTreeClassifier` полностью: предсказания на обучающей и тестовой выборках, метрики и структура (признак и порог в каждом узле) идентичны для всех проверенных наборов параметров, а также на 20 других сгенерированных наборах данных.

**Оговорка о точности совпадения.** Если в узле есть два разных разбиения с в точности равным приростом информации (например, каждое отделяет по одному объекту одного и того же класса, что типично для почти чистых узлов при `min_samples_leaf=1`), то sklearn выбирает среди них по случайному порядку перебора признаков (зависит от `random_state`), а самописное — первое найденное. Тогда деревья могут отличаться, хотя качество у них одинаковое. Поэтому для сравнения выбраны параметры с `min_samples_leaf` ≥ 10: единичные объекты в таких условиях не отделяются, и подобные ничьи редки (при `min_samples_leaf=1` и полностью выросшем дереве расхождения в структуре встречаются часто, хотя качество остаётся сопоставимым).

## Общий вывод

В лабораторной реализовано дерево решений CART: критерий Джини, прирост информации, четыре критерия останова и метрики качества. Результаты самописной модели совпадают с `DecisionTreeClassifier` из sklearn.